# CrewAI: Building Multi-Agent AI Systems

A content-creation pipeline built with **CrewAI**: a research agent gathers current information with a live web-search tool, then a writer agent turns those findings into an article. The two run as a crew, passing work between them in sequence.

Covers the three building blocks — **agents** (who does the work), **tasks** (what to do), and the **crew** (how they run together) — and how to read back results, per-task outputs, and token usage.

## Setup


Install dependencies.

In [ ]:
%%capture
%pip install crewai crewai-tools python-dotenv

## What CrewAI gives you

- **Agent** — a role, a goal, and a backstory that shapes how it behaves
- **Task** — a unit of work assigned to an agent, with an expected output
- **Crew** — the agents and tasks together, run sequentially or in parallel
- **Tools** — capabilities an agent can call, such as web search

## Web search tool

Agents need current information, so the research agent gets a Serper-backed Google Search tool. Without it the agent could only draw on what the model already knows.

Load the API keys.

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv

# Keys come from a .env file - never hard-code them in the notebook.
load_dotenv(find_dotenv(usecwd=True))

for var in ("GROQ_API_KEY", "SERPER_API_KEY"):
    if not os.environ.get(var):
        raise RuntimeError(f"{var} is not set. Add it to your .env file.")

print("API keys loaded.")

Import the search tool.

In [ ]:
%%capture

from crewai_tools import SerperDevTool

Initialize it.

In [ ]:
search_tool=SerperDevTool()
print(type(search_tool))

Try a search directly, before wiring it to an agent.

In [ ]:
search_query = "Latest Breakthroughs in machine learning"
search_results =search_tool.run(query=search_query )

# Print the results
print(f"Search Results for '{search_results}':\n")

The response holds several keys; `organic` has the ranked results.

In [ ]:
print("keys of search_results", search_results.keys())

## The model

One model instance shared by every agent in the crew.

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from crewai import LLM

# Load GROQ_API_KEY (and SERPER_API_KEY where needed) from a .env file.
load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError(
        "GROQ_API_KEY is not set. Copy .env.example to .env and add your key "
        "(free at https://console.groq.com/keys)."
    )

# Groq through its OpenAI-compatible endpoint.
llm = LLM(
    model="openai/llama-3.3-70b-versatile",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    max_tokens=2000,
)

## Agents

An agent is defined by its **role**, **goal**, and **backstory** — together these steer how it interprets a task. `tools` gives it capabilities, and `allow_delegation` controls whether it can hand work to another agent.

### The research agent

In [ ]:
from crewai import Agent

research_agent = Agent(
  role='Senior Research Analyst',
  goal='Uncover cutting-edge information and insights on any subject with comprehensive analysis',
  backstory="""You are an expert researcher with extensive experience in gathering, analyzing, and synthesizing information across multiple domains. 
  Your analytical skills allow you to quickly identify key trends, separate fact from opinion, and produce insightful reports on any topic. 
  You excel at finding reliable sources and extracting valuable information efficiently.""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[SerperDevTool()]
)

In [ ]:
research_agent

### The writer agent

A second specialist, without search access — it works from what the researcher produces.

In [ ]:
# Define your agents with roles and goals
# Define the Writer Agent
writer_agent = Agent(
  role='Tech Content Strategist',
  goal='Craft well-structured and engaging content based on research findings',
  backstory="""You are a skilled content strategist known for translating 
  complex topics into clear and compelling narratives. Your writing makes 
  information accessible and engaging for a wide audience.""",
  verbose=True,
  llm = llm,
  allow_delegation=True
)

In [ ]:
writer_agent 

## Tasks

A task pairs a description with the agent that runs it and the output expected.

### Research task

In [ ]:
from crewai import Task

research_task = Task(
  description="Analyze the major {topic}, identifying key trends and technologies. Provide a detailed report on their potential impact.",
  agent=research_agent,
  expected_output="A detailed report on {topic}, including trends, emerging technologies, and their impact."
)

### Writer task

Takes the research findings and produces the article.

In [ ]:
# Create a task for the Writer Agent
writer_task = Task(
  description="Create an engaging blog post based on the research findings about {topic}. Tailor the content for a tech-savvy audience, ensuring clarity and interest.",
  agent=writer_agent,
  expected_output="A 4-paragraph blog post on {topic}, written clearly and engagingly for tech enthusiasts."
)

## The crew

Group the agents and tasks. `Process.sequential` runs tasks in order, each receiving the previous output.

In [ ]:
from crewai import Crew, Process

crew = Crew(
    agents=[research_agent, writer_agent],
    tasks=[research_task, writer_task],
    process=Process.sequential,
    verbose=True 
)

`kickoff()` runs the crew. `inputs` fills the `{topic}` placeholder in the task descriptions.

In [ ]:
result = crew.kickoff(inputs={"topic": "Latest Generative AI breakthroughs"})

The result is a `CrewOutput` object.

In [ ]:
type(result)

In [ ]:
result

`result.raw` is the final text.

In [ ]:
final_output = result.raw
print("Final output:", final_output)

`tasks_output` exposes each task's result individually.

In [ ]:
tasks_outputs = result.tasks_output

Research task description and output.

In [ ]:
print("Task Description", tasks_outputs[0].description)
print("Output of research task ",tasks_outputs[0])

Writer task description and output.

In [ ]:
print("Writer task description:", tasks_outputs[1].description)
print(" \nOutput of writer task:", tasks_outputs[1].raw)

Each task output also records which agent produced it.

In [ ]:
print("We can get the agent for researcher task:  ",tasks_outputs[0].agent)
print("We can get the agent for the writer task: ",tasks_outputs[1].agent)

Token usage for the whole run.

In [ ]:
token_count = result.token_usage.total_tokens
prompt_tokens = result.token_usage.prompt_tokens
completion_tokens = result.token_usage.completion_tokens

print(f"Total tokens used: {token_count}")
print(f"Prompt tokens: {prompt_tokens} (used for instructions to the model)")
print(f"Completion tokens: {completion_tokens} (generated in response)")

## Extending the crew: a social media strategist

Add a third agent that turns the article into social posts.

In [ ]:
social_agent = Agent(
    role='Social Media Strategist',
    goal='Generate engaging social media snippets based on the full article',
    backstory="A digital storyteller who excels at crafting compelling posts to drive engagement and traffic.",
    verbose=True
)

Its task.

In [ ]:
social_task = Task(
    description=(
        "Summarize the blog post about {topic} into 2–3 engaging social media posts "
        "suitable for platforms like LinkedIn or Twitter. Make sure the tone is informative, "
        "professional, and encourages further reading."
    ),
    agent=social_agent,
    expected_output="A series of 2–3 well-written social posts highlighting the key insights from the blog content."
)

Run all three agents as one crew.

In [ ]:
crew = Crew(
    agents=[research_agent, writer_agent, social_agent],
    tasks=[research_task, writer_task, social_task],
    process=Process.sequential,  # Tasks will be executed one after another
    verbose=True
)

# Run the crew and capture the final output (includes research, blog post, and social media content)
result = crew.kickoff(inputs={"topic": "Latest Generative AI breakthroughs"})

## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)